<a href="https://colab.research.google.com/github/1HPz/Super-AI-Engineer-Season-6/blob/main/OCR_EasyOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. ลบของเก่าและลงเวอร์ชันที่เสถียร (Fix ImportError: _Ink)
!pip uninstall -y Pillow
!pip install Pillow==10.2.0
!pip install easyocr pandas thefuzz

import os
import glob
import re
import pandas as pd
import easyocr
from thefuzz import process
from PIL import Image

# 2. โหลดโมเดล (Pre-trained)
# ครั้งแรกจะดาวน์โหลดไฟล์นิดนึงครับ
reader = easyocr.Reader(['th', 'en'])

# 3. ตั้งค่าไฟล์
TEMPLATE_PATH = 'submission_template_v3.csv'
IMAGE_FOLDER = '/content/drive/MyDrive/Image'

# โหลดรายชื่อพรรค
df_temp = pd.read_csv(TEMPLATE_PATH)
df_temp = df_temp.rename(columns={df_temp.columns[1]: 'party_name'})
all_party_names = df_temp['party_name'].unique().tolist()

print("--- ✅ แก้ไขและติดตั้งสำเร็จ! กรุณา Restart Session หากยังติด Error เดิม ---")

Found existing installation: pillow 12.1.1
Uninstalling pillow-12.1.1:
  Successfully uninstalled pillow-12.1.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 71.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete--- ✅ แก้ไขและติดตั้งสำเร็จ! กรุณา Restart Session หากยังติด Error เดิม ---


In [ ]:
extracted_data = []
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.*")))

print(f"พบรูปภาพทั้งหมด {len(image_paths)} รูป")

for i, path in enumerate(image_paths):
    fname = os.path.basename(path)
    clean_name = os.path.splitext(fname)[0]
    parts = clean_name.split("_")
    prefix = "_".join(parts[:4]) if fname.startswith("party_list") else "_".join(parts[:3])

    print(f"[{i+1}/{len(image_paths)}] EasyOCR กำลังอ่าน: {fname}")

    # อ่านข้อความทั้งหมดในรูป
    # detail=0 จะคืนค่ามาแค่ List ของข้อความที่อ่านได้
    results = reader.readtext(path, detail=0)

    current_party = None

    for text in results:
        # 1. หาชื่อพรรคที่ใกล้เคียงใน Template
        match_name, score = process.extractOne(text, all_party_names)

        if score > 80:
            current_party = match_name
            continue # เจอชื่อพรรคแล้ว ไปหาตัวเลขในบรรทัดถัดไป

        # 2. ถ้าเจอตัวเลขหลังจากเจอชื่อพรรค ให้ถือว่าเป็นคะแนน
        if current_party:
            # ใช้ Regex ดึงเฉพาะตัวเลข (เผื่อ AI อ่านติดอักขระอื่นมา)
            votes_match = re.findall(r'\d+', text)
            if votes_match:
                votes_value = int("".join(votes_match))
                extracted_data.append({
                    'prefix': prefix,
                    'party_name': current_party,
                    'votes': votes_value
                })
                print(f"   ✅ แมปได้: {current_party} = {votes_value}")
                current_party = None # รีเซ็ตเพื่อหาพรรคถัดไป

print(f"--- เซลล์ที่ 2: สกัดข้อมูลได้ทั้งหมด {len(extracted_data)} รายการ ---")

พบรูปภาพทั้งหมด 846 รูป
[1/846] EasyOCR กำลังอ่าน: constituency_10_1.png
   ✅ แมปได้: รวมพลัง = 1
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 12
   ✅ แมปได้: ภูมิใจไทย = 82421
   ✅ แมปได้: รักชาติ = 2
   ✅ แมปได้: รวมพลัง = 2
   ✅ แมปได้: รักชาติ = 23
   ✅ แมปได้: ฟิวชัน = 31
   ✅ แมปได้: บ้านเมือง = 78649
   ✅ แมปได้: ไทยก้าวหน้า = 32
   ✅ แมปได้: บ้านเมือง = 33
   ✅ แมปได้: บ้านเมือง = 34
   ✅ แมปได้: รักชาติ = 41
   ✅ แมปได้: บ้านเมือง = 42
   ✅ แมปได้: บ้านเมือง = 43
   ✅ แมปได้: บ้านเมือง = 4200
[2/846] EasyOCR กำลังอ่าน: constituency_10_10.png


   ✅ แมปได้: สังคมประชาธิปไตยไทย = 10
   ✅ แมปได้: บ้านเมือง = 10
   ✅ แมปได้: รวมพลัง = 1
   ✅ แมปได้: ภูมิใจไทย = 137981
   ✅ แมปได้: บ้านเมือง = 12
   ✅ แมปได้: ภูมิใจไทย = 99657
   ✅ แมปได้: กรีน = 2
   ✅ แมปได้: รวมพลัง = 21
   ✅ แมปได้: ฟิวชัน = 22
   ✅ แมปได้: กรีน = 23
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 3
   ✅ แมปได้: ฟิวชัน = 31
   ✅ แมปได้: บ้านเมือง = 94839
   ✅ แมปได้: ไทยก้าวหน้า = 32
   ✅ แมปได้: บ้านเมือง = 9
   ✅ แมปได้: บ้านเมือง = 488
   ✅ แมปได้: กรีน = 4
   ✅ แมปได้: บ้านเมือง = 91913
   ✅ แมปได้: บ้านเมือง = 6223
[3/846] EasyOCR กำลังอ่าน: constituency_10_10_page2.png
   ✅ แมปได้: ครูไทยเพื่อประชาชน = 6
   ✅ แมปได้: ประชาชน = 41814
   ✅ แมปได้: โอกาสใหม่ = 9440
   ✅ แมปได้: รักชาติ = 3
   ✅ แมปได้: ประชาธิปัตย์ = 6372
   ✅ แมปได้: รวมไทยสร้างชาติ = 2012
   ✅ แมปได้: ไทยสร้างไทย = 636
   ✅ แมปได้: ไทยก้าวใหม่ = 583
   ✅ แมปได้: กล้าธรรม = 545
   ✅ แมปได้: ประชาธิปไตยใหม่ = 503
   ✅ แมปได้: พลวัต = 461
   ✅ แมปได้: รักชาติ = 3
   ✅ แมปได้: ประชากรไทย = 282
   ✅ แมปได

   ✅ แมปได้: รวมพลัง = 3
[10/846] EasyOCR กำลังอ่าน: constituency_10_13.png
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 13
   ✅ แมปได้: วิชชั่นใหม่ = 8
   ✅ แมปได้: เป็นธรรม = 13
   ✅ แมปได้: รวมพลัง = 1
   ✅ แมปได้: ภูมิใจไทย = 1
   ✅ แมปได้: ภูมิใจไทย = 131666
   ✅ แมปได้: กรีน = 12
   ✅ แมปได้: ฟิวชัน = 2
   ✅ แมปได้: รวมพลัง = 21
   ✅ แมปได้: บ้านเมือง = 22
   ✅ แมปได้: บ้านเมือง = 23
   ✅ แมปได้: บ้านเมือง = 3
   ✅ แมปได้: ฟิวชัน = 31
   ✅ แมปได้: บ้านเมือง = 32
   ✅ แมปได้: บ้านเมือง = 33
   ✅ แมปได้: บ้านเมือง = 34
   ✅ แมปได้: เศรษฐกิจ = 41
   ✅ แมปได้: บ้านเมือง = 42
   ✅ แมปได้: บ้านเมือง = 3
   ✅ แมปได้: บ้านเมือง = 5
[11/846] EasyOCR กำลังอ่าน: constituency_10_13_page2.png
   ✅ แมปได้: ประชาชน = 44511
   ✅ แมปได้: มิติใหม่ = 15227
   ✅ แมปได้: ประชาธิปัตย์ = 11428
   ✅ แมปได้: เพื่อไทย = 10752
   ✅ แมปได้: รวมไทยสร้าง = 887
   ✅ แมปได้: เศรษฐกิจ = 413
   ✅ แมปได้: ปวงชนไทย = 108
   ✅ แมปได้: ไทยสร้างไทย = 807
   ✅ แมปได้: ใหม่ = 775
   ✅ แมปได้: พลวัต = 325
   ✅ แมปได้: พลังประชารัฐ

   ✅ แมปได้: ไทยพิทักษ์ธรรม = 14
   ✅ แมปได้: ประชาชน = 41
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 13
   ✅ แมปได้: ภูมิใจไทย = 25491
   ✅ แมปได้: เพื่อไทย = 10848
   ✅ แมปได้: ประชาธิปัตย์ = 886
   ✅ แมปได้: ไทยสร้างชาติ = 1333
   ✅ แมปได้: ไทยสร้างไทย = 1299
   ✅ แมปได้: ไทยทรัพย์ทวี = 1199
   ✅ แมปได้: ไทยภักดี = 694
   ✅ แมปได้: กล้าธรรม = 571
   ✅ แมปได้: ไทยก้าวใหม่ = 485
   ✅ แมปได้: รักชาติ = 477
   ✅ แมปได้: รักชาติ = 2
   ✅ แมปได้: ปวงชนไทย = 354
   ✅ แมปได้: พลังประชารัฐ = 352
   ✅ แมปได้: โอกาสใหม่ = 291
   ✅ แมปได้: พลวัต = 213
   ✅ แมปได้: ไทยก้าวหน้า = 146
   ✅ แมปได้: ฟิวชัน = 93467
   ✅ แมปได้: ภูมิใจไทย = 528
[14/846] EasyOCR กำลังอ่าน: constituency_10_14_page3.png


   ✅ แมปได้: รวมไทยสร้างชาติ = 56
   ✅ แมปได้: กล้าธรรม = 2
   ✅ แมปได้: ภูมิใจไทย = 17
   ✅ แมปได้: เพื่อไทย = 4
   ✅ แมปได้: ประชาธิปัตย์ = 7
   ✅ แมปได้: โอกาสใหม่ = 5
   ✅ แมปได้: รวมพลัง = 33
   ✅ แมปได้: รวมพลัง = 4
   ✅ แมปได้: รวมพลัง = 4
   ✅ แมปได้: โอกาสใหม่ = 1
[15/846] EasyOCR กำลังอ่าน: constituency_10_16.png
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 16
   ✅ แมปได้: วิชชั่นใหม่ = 8
   ✅ แมปได้: เป็นธรรม = 16
   ✅ แมปได้: ภูมิใจไทย = 11
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 12
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 2
   ✅ แมปได้: รวมพลัง = 21
   ✅ แมปได้: บ้านเมือง = 23
   ✅ แมปได้: บ้านเมือง = 3
   ✅ แมปได้: ฟิวชัน = 31
   ✅ แมปได้: บ้านเมือง = 33
   ✅ แมปได้: บ้านเมือง = 4561
   ✅ แมปได้: บ้านเมือง = 4
   ✅ แมปได้: บ้านเมือง = 43
[16/846] EasyOCR กำลังอ่าน: constituency_10_16_page2.png


...

   ✅ แมปได้: บ้านเมือง = 38
   ✅ แมปได้: พลังธรรมใหม่ = 260
   ✅ แมปได้: กรีน = 55
   ✅ แมปได้: ไทยธรรม = 129
   ✅ แมปได้: แผ่นดินธรรม = 43
   ✅ แมปได้: กล้าธรรม = 1630
   ✅ แมปได้: บ้านเมือง = 3
   ✅ แมปได้: พลังประชารัฐ = 3
   ✅ แมปได้: โอกาสใหม่ = 36
   ✅ แมปได้: เป็นธรรม = 47
   ✅ แมปได้: ประชาชน = 13876
   ✅ แมปได้: ประชาไทย = 145
   ✅ แมปได้: ไทยสร้างไทย = 3
   ✅ แมปได้: ไทยก้าวใหม่ = 107
   ✅ แมปได้: ประชาอาสาชาติ = 27
   ✅ แมปได้: ไทยก้าวหน้า = 3
   ✅ แมปได้: ไทยพิทักษ์ธรรม = 4
   ✅ แมปได้: ไทยรวมไทย = 17
   ✅ แมปได้: เพื่อบ้านเมือง = 38
   ✅ แมปได้: พลังไทยรักชาติ = 51
   ✅ แมปได้: ไทยก้าวหน้า = 9
   ✅ แมปได้: รวมพลัง = 4
   ✅ แมปได้: รวมพลัง = 17
[820/846] EasyOCR กำลังอ่าน: party_list_33_5.png
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 5
   ✅ แมปได้: กรีน = 5
   ✅ แมปได้: เพื่อชีวิตใหม่ = 1
   ✅ แมปได้: ภูมิใจไทย = 11
   ✅ แมปได้: ภูมิใจไทย = 127679
   ✅ แมปได้: ไทยก้าวหน้า = 12
   ✅ แมปได้: ภูมิใจไทย = 85063
   ✅ แมปได้: รวมพลัง = 21
   ✅ แมปได้: รักชาติ = 130792
   ✅ แมปได้: เพื่อ

   ✅ แมปได้: ไทยพร้อม = 351
   ✅ แมปได้: ภูมิใจไทย = 30867
   ✅ แมปได้: พลังธรรมใหม่ = 443
   ✅ แมปได้: รักชาติ = 39
   ✅ แมปได้: กรีน = 67
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 41
   ✅ แมปได้: แผ่นดินธรรม = 47
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 42
   ✅ แมปได้: กล้าธรรม = 100
   ✅ แมปได้: โอกาสใหม่ = 32
   ✅ แมปได้: เป็นธรรม = 55
   ✅ แมปได้: ไทยก้าวหน้า = 47
   ✅ แมปได้: ประชาไทย = 158
   ✅ แมปได้: บ้านเมือง = 49
   ✅ แมปได้: ไทยก้าวใหม่ = 166
   ✅ แมปได้: ประชาอาสาชาติ = 14
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 51
   ✅ แมปได้: พร้อม = 3
   ✅ แมปได้: เครือข่ายชาวนาแห่งประเทศไทย = 10
   ✅ แมปได้: บ้านเมือง = 53
   ✅ แมปได้: ไทยพิทักษ์ธรรม = 12
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 54
   ✅ แมปได้: ความหวังใหม่ = 5
   ✅ แมปได้: ไทยก้าวหน้า = 55
   ✅ แมปได้: ไทยรวมไทย = 13
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 57
   ✅ แมปได้: พลังไทยรักชาติ = 51
   ✅ แมปได้: ฟิวชัน = 83511
   ✅ แมปได้: ไทยภักดี = 7
[833/846] EasyOCR กำลังอ่าน: party_list_33_9.png
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 9
   ✅ แมปได้: บ้าน

   ✅ แมปได้: รวมใจไทย = 466
   ✅ แมปได้: รวมไทยสร้างชาติ = 732
   ✅ แมปได้: พลวัต = 66
   ✅ แมปได้: ประชาธิปไตยใหม่ = 375
   ✅ แมปได้: เพื่อไทย = 33299
   ✅ แมปได้: ทางเลือกใหม่ = 2208
   ✅ แมปได้: เสรีรวมไทย = 329
   ✅ แมปได้: ไทยก้าวหน้า = 13
   ✅ แมปได้: รวมพลังประชาชน = 405
   ✅ แมปได้: ท้องที่ไทย = 138
   ✅ แมปได้: อนาคตไทย = 48
   ✅ แมปได้: พลังเพื่อไทย = 0
   ✅ แมปได้: ไทยชนะ = 112
   ✅ แมปได้: พลังสังคมใหม่ = 24
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 19
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 29
   ✅ แมปได้: ฟิวชัน = 22
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 21
   ✅ แมปได้: ไทรวมพลัง = 107
   ✅ แมปได้: ก้าวอิสระ = 26
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 23
   ✅ แมปได้: ปวงชนไทย = 44
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 24
   ✅ แมปได้: วิชชั่นใหม่ = 72
   ✅ แมปได้: เพื่อชีวิตใหม่ = 26
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 6
   ✅ แมปได้: คลองไทย = 27
   ✅ แมปได้: ประชาธิปัตย์ = 1033
   ✅ แมปได้: ไทยก้าวหน้า = 29
   ✅ แมปได้: ไทยภักดี = 203
   ✅ แมปได้: แรงงานสร้างชาติ = 3
   ✅ แมปได้: ประชากรไทย = 4

   ✅ แมปได้: วิชชั่นใหม่ = 8
   ✅ แมปได้: บ้านเมือง = 1
   ✅ แมปได้: เพื่อชีวิตใหม่ = 1
   ✅ แมปได้: ภูมิใจไทย = 1
   ✅ แมปได้: ภูมิใจไทย = 132480
   ✅ แมปได้: รักชาติ = 12
   ✅ แมปได้: ภูมิใจไทย = 99451
   ✅ แมปได้: รักชาติ = 2
   ✅ แมปได้: รวมพลัง = 21
   ✅ แมปได้: รักชาติ = 142876
   ✅ แมปได้: รักชาติ = 23
   ✅ แมปได้: ฟิวชัน = 31
   ✅ แมปได้: บ้านเมือง = 91635
   ✅ แมปได้: ไทยก้าวหน้า = 32
   ✅ แมปได้: บ้านเมือง = 33
   ✅ แมปได้: บ้านเมือง = 7811
   ✅ แมปได้: รวมไทยสร้างชาติ = 99451
   ✅ แมปได้: รักชาติ = 41
   ✅ แมปได้: บ้านเมือง = 42
   ✅ แมปได้: บ้านเมือง = 43
[837/846] EasyOCR กำลังอ่าน: party_list_34_10.png


   ✅ แมปได้: สังคมประชาธิปไตยไทย = 10
   ✅ แมปได้: บ้านเมือง = 10
   ✅ แมปได้: เพื่อชีวิตใหม่ = 1
   ✅ แมปได้: ภูมิใจไทย = 11
   ✅ แมปได้: ภูมิใจไทย = 141083
   ✅ แมปได้: ภูมิใจไทย = 93234
   ✅ แมปได้: รวมพลัง = 21
   ✅ แมปได้: รักชาติ = 138236
   ✅ แมปได้: ฟิวชัน = 31
   ✅ แมปได้: บ้านเมือง = 87492
   ✅ แมปได้: รักชาติ = 32
   ✅ แมปได้: รวมพลัง = 33
   ✅ แมปได้: บ้านเมือง = 34
   ✅ แมปได้: รวมไทยสร้างชาติ = 93234
[838/846] EasyOCR กำลังอ่าน: party_list_34_10_page2.png
   ✅ แมปได้: สังคมประชาธิปไตยไทย = 41
   ✅ แมปได้: ไทยก้าวหน้า = 43
   ✅ แมปได้: บ้านเมือง = 1497
   ✅ แมปได้: รักชาติ = 5
   ✅ แมปได้: ไทยทรัพย์ทวี = 66
   ✅ แมปได้: ไทยก้าวหน้า = 3
   ✅ แมปได้: ใหม่ = 743
   ✅ แมปได้: มิติใหม่ = 189
   ✅ แมปได้: ไทยก้าวหน้า = 5
   ✅ แมปได้: รวมใจไทย = 680
   ✅ แมปได้: รวมไทยสร้างชาติ = 759
   ✅ แมปได้: ไทยก้าวหน้า = 75
   ✅ แมปได้: ประชาธิปไตยใหม่ = 229
   ✅ แมปได้: เพื่อไทย = 9462
   ✅ แมปได้: ทางเลือกใหม่ = 267
   ✅ แมปได้: เศรษฐกิจ = 2896
   ✅ แมปได้: เสรีรวมไทย = 616
   ✅ แมปได้: ร

In [ ]:
# 1. รวมคะแนน (กรณีพรรคเดียวมีหลายหน้า)
df_raw = pd.DataFrame(extracted_data)
if not df_raw.empty:
    df_summed = df_raw.groupby(['prefix', 'party_name'])['votes'].sum().reset_index()
else:
    df_summed = pd.DataFrame(columns=['prefix', 'party_name', 'votes'])

# 2. เตรียม Template
df_final = pd.read_csv(TEMPLATE_PATH)
df_final = df_final.rename(columns={df_final.columns[1]: 'party_name'})

def make_prefix(id_str):
    parts = id_str.split("_")
    return "_".join(parts[:4]) if id_str.startswith("party_list") else "_".join(parts[:3])

df_final['prefix'] = df_final['id'].apply(make_prefix)

# 3. Merge ข้อมูล
df_output = df_final.merge(df_summed, on=['prefix', 'party_name'], how='left', suffixes=('', '_new'))

# 4. จัดการค่าว่างเป็น 0 และเลือก 2 คอลัมน์เป๊ะๆ
df_output['votes'] = df_output['votes_new'].fillna(0).astype(int)
df_final_result = df_output[['id', 'votes']]

# 5. บันทึกไฟล์
output_name = 'submission_easyocr.csv'
df_final_result.to_csv(output_name, index=False)

print(f"--- ✅ สำเร็จ! ไฟล์ {output_name} พร้อมส่งแล้ว (2 คอลัมน์) ---")
print(df_final_result.head(10))

--- ✅ สำเร็จ! ไฟล์ submission_easyocr.csv พร้อมส่งแล้ว (2 คอลัมน์) ---
                     id  votes
0   constituency_10_1_1  14813
1   constituency_10_1_2  96789
2   constituency_10_1_3    979
3   constituency_10_1_4    244
4   constituency_10_1_5     35
5   constituency_10_1_6  34167
6   constituency_10_1_7      0
7   constituency_10_1_8   1023
8   constituency_10_1_9      0
9  constituency_10_1_10     68
